# APUSH FRQ Grader v5 — final Colab run

Merge v4, train separate score and feedback adapters, compare on private development data, freeze configuration, and evaluate once on the 53-case CB-derived golden set. Private rows and per-case predictions must not be redistributed. Because capped golden excerpts informed synthetic style, the golden evaluation is **development-informed**, not untouched.

## 1. GPU, frozen code, dependencies, and Drive

In [ ]:
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Attach a GPU runtime.'
REPO=Path('/content/slm'); REF=os.environ.get('SLM_GIT_REF','v5')
if not (REPO/'.git').exists(): subprocess.run(['git','clone','https://github.com/aryanjverma/slm.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin'],cwd=REPO,check=True)
subprocess.run(['git','checkout',REF],cwd=REPO,check=True)
if REF=='v5': subprocess.run(['git','pull','--ff-only','origin','v5'],cwd=REPO,check=True)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train]','sentencepiece','tiktoken'],check=True)
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/slm_v5')
PRIVATE=Path(os.environ.get('V5_PRIVATE_ROOT',str(ROOT/'private_repo/artifacts/data/v5')))
V4=Path(os.environ.get('V4_ADAPTER_DIR','/content/drive/MyDrive/slm_v4/apush-frq-grader-v4-assistant-only-r1/final'))
MERGED=ROOT/'merged_v4_base'; SCORER_RUN=ROOT/'scorer/final-r1'; FEEDBACK_RUN=ROOT/'feedback/final-r1'; EVAL=ROOT/'eval/final-r1'
for path in (ROOT,EVAL): path.mkdir(parents=True,exist_ok=True)
print(torch.cuda.get_device_name(0),REVISION)

## 2. Private preflight and approved 615-row runtime set

The finalized private directory contains 540 new train cases, 60 development cases, and 75 balanced v4 replay cases. Approval is bound to the reviewed 60-case packet. This cell verifies counts and receipts, then creates a private Drive-only combined training JSONL; it never prints or embeds rows.

In [ ]:
MANIFEST=PRIVATE/'private_use_manifest_v5.json'
AUDIT=PRIVATE/'assembly_audit_v5.json'
APPROVAL=PRIVATE/'manual_review_approval_v5.json'
PACKET=PRIVATE/'manual_review_packet_v5.jsonl'
NEW_TRAIN=PRIVATE/'train_cases_v5.jsonl'
DEV=PRIVATE/'dev_cases_v5.jsonl'
REPLAY=PRIVATE/'replay_cases_v4_for_v5.jsonl'
TRAIN=ROOT/'private_runtime/train_cases_v5_with_replay.jsonl'
def sha(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1048576),b''): d.update(chunk)
 return d.hexdigest()
def count(path):
 with Path(path).open(encoding='utf-8') as f: return sum(bool(line.strip()) for line in f)
for path in (MANIFEST,AUDIT,APPROVAL,PACKET,NEW_TRAIN,DEV,REPLAY): assert path.exists(),f'Missing {path}'
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
audit=json.loads(AUDIT.read_text(encoding='utf-8'))
approval=json.loads(APPROVAL.read_text(encoding='utf-8'))
assert manifest['selected']==600 and manifest['train']==540 and manifest['dev']==60 and manifest['manual_review']==60
assert approval.get('approved') is True and approval.get('reviewer') and approval.get('approved_at')
assert approval.get('packet_sha256')==sha(PACKET),'Review packet changed after approval'
assert count(PACKET)==60 and count(NEW_TRAIN)==540 and count(DEV)==60 and count(REPLAY)==75
for key,value in {'approved':True,'new_train':540,'new_dev':60,'v4_replay':75,'training_rows_total':615,'golden_eval_rows_in_training':0}.items(): assert audit.get(key)==value,(key,audit.get(key))
for path in (NEW_TRAIN,DEV,REPLAY): assert audit['artifacts'][path.name]==sha(path),f'Hash mismatch: {path}'
TRAIN.parent.mkdir(parents=True,exist_ok=True)
with TRAIN.open('w',encoding='utf-8') as out:
 for source in (NEW_TRAIN,REPLAY):
  with source.open(encoding='utf-8') as incoming:
   for line in incoming:
    if line.strip(): out.write(line.rstrip()+ '\n')
assert count(TRAIN)==615
print({'train':615,'development':60,'manual_review':60,'approved':True})

## 3. Merge the completed v4 adapter in FP16

The inherited-base receipt hashes both the source adapter and merged model. Reruns skip a complete merge.

In [ ]:
BASE='Qwen/Qwen2.5-0.5B-Instruct'
MERGE=REPO/'scripts/merge_v4_adapter.py'; MERGED_META=MERGED/'v5_inherited_base.json'
assert MERGE.exists() and (V4/'adapter_config.json').exists()
if not MERGED_META.exists():
 subprocess.run([sys.executable,str(MERGE),'--v4-adapter',str(V4),'--base-model',BASE,'--output',str(MERGED),'--no-bf16'],check=True)
merged_meta=json.loads(MERGED_META.read_text(encoding='utf-8'))
assert merged_meta['v4_adapter_sha256'] and merged_meta['inherited_base_sha256']
print('Inherited base:',merged_meta['inherited_base_sha256'])

## 4. Train scorer and score-conditioned feedback adapters

Both are fresh rank-16 QLoRA adapters with 4,096-token context and resume support. The scorer weights numeric tokens 4x and uses four epochs at 1e-4. Feedback uses two epochs at 5e-5. Checkpoints are evaluated every 100 steps with patience two.

In [ ]:
COMMON={'max_seq_length':4096,'lora_rank':16,'batch_size':1,'grad_accum':4,'warmup_ratio':.03,'eval_steps':100,'early_stopping_patience':2,'seed':13}
CONFIG={'scorer':{'epochs':4,'learning_rate':1e-4,'score_token_weight':4.0},'feedback':{'epochs':2,'learning_rate':5e-5}}
TRAIN_SCRIPT=REPO/'scripts/train_v5.py'
def latest(run):
 good=[x for x in run.glob('checkpoint-*') if (x/'trainer_state.json').exists()]
 return max(good,key=lambda x:int(x.name.rsplit('-',1)[-1])) if good else None
def train(task,run):
 final=run/'final'
 if (final/'adapter_config.json').exists(): return
 cfg=CONFIG[task]
 cmd=[sys.executable,str(TRAIN_SCRIPT),'--task',task,'--model',str(MERGED),'--data',str(TRAIN),'--eval-data',str(DEV),'--output',str(run),'--max-seq-length',str(COMMON['max_seq_length']),'--lora-rank',str(COMMON['lora_rank']),'--batch-size','1','--grad-accum','4','--warmup-ratio','.03','--eval-steps','100','--early-stopping-patience','2','--seed','13','--epochs',str(cfg['epochs']),'--learning-rate',str(cfg['learning_rate'])]
 if task=='scorer': cmd += ['--score-token-weight','4']
 checkpoint=latest(run)
 if checkpoint: cmd += ['--resume-from-checkpoint',str(checkpoint)]
 log=ROOT/f'v5_{task}_final-r1.log'
 with log.open('a',encoding='utf-8') as stream:
  proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
  for line in proc.stdout: print(line,end=''); stream.write(line); stream.flush()
  assert proc.wait()==0,f'{task} failed; inspect {log}'
train('scorer',SCORER_RUN); train('feedback',FEEDBACK_RUN)
SCORER=SCORER_RUN/'final'; FEEDBACK=FEEDBACK_RUN/'final'
for path in (SCORER,FEEDBACK):
 assert (path/'adapter_config.json').exists() and (path/'v5_training_metadata.json').exists()

## 5. Package and run a ten-case two-pass smoke evaluation

The bundle manifest locks the inherited base and both adapters. Evaluation exercises validated scores, deterministic totals, grounded feedback, and fallback feedback without printing private cases.

In [ ]:
PACKAGE=REPO/'scripts/package_v5_bundle.py'; GRADE_SCRIPT=REPO/'scripts/grade_v5.py'; EVAL_SCRIPT=REPO/'scripts/eval_v5.py'
assert PACKAGE.exists() and GRADE_SCRIPT.exists() and EVAL_SCRIPT.exists()
def adapter_candidates(run):
 paths=sorted((p for p in run.glob('checkpoint-*') if (p/'adapter_config.json').exists()),key=lambda p:int(p.name.rsplit('-',1)[-1]))
 paths.append(run/'final')
 return list(dict.fromkeys(path.resolve() for path in paths))
def temporary_eval(label,scorer,feedback):
 bundle=ROOT/'checkpoint_selection/bundles'/label; out=ROOT/'checkpoint_selection/eval'/label
 subprocess.run([sys.executable,str(PACKAGE),'--bundle',str(bundle),'--inherited-base',str(MERGED),'--scorer',str(scorer),'--feedback',str(feedback),'--reference-only','--overwrite'],check=True)
 subprocess.run([sys.executable,str(EVAL_SCRIPT),'--bundle',str(bundle),'--eval-path',str(DEV),'--output-dir',str(out),'--model-name',label,'--bootstrap-samples','100','--no-verify-hashes','--resume'],check=True)
 return json.loads((out/f'{label}_real_summary.json').read_text(encoding='utf-8'))
scorer_rank=[]
for index,path in enumerate(adapter_candidates(SCORER_RUN)):
 summary=temporary_eval(f'scorer-{index:02d}',path,FEEDBACK)
 scorer_rank.append({'adapter':str(path),'summary':summary})
def scorer_key(item):
 s=item['summary']; qwk=s['qwk'] if s['qwk'] is not None else -1.0; exact=s['criterion_exact_rates']
 return (qwk,-s['total_mae'],exact['evidence'],sum(exact.values())/len(exact))
scorer_rank.sort(key=scorer_key,reverse=True); SCORER=Path(scorer_rank[0]['adapter'])
feedback_rank=[]
for index,path in enumerate(adapter_candidates(FEEDBACK_RUN)):
 summary=temporary_eval(f'feedback-{index:02d}',SCORER,path)
 feedback_rank.append({'adapter':str(path),'summary':summary})
def feedback_key(item):
 s=item['summary']; return (s['evidence_grounding_rate'],s['structured_output_valid_rate'],-s['feedback_fallback_rate'])
feedback_rank.sort(key=feedback_key,reverse=True); FEEDBACK=Path(feedback_rank[0]['adapter'])
SELECTION=ROOT/'checkpoint_selection/selected.json'; SELECTION.parent.mkdir(parents=True,exist_ok=True)
SELECTION.write_text(json.dumps({'scorer':scorer_rank,'feedback':feedback_rank,'selected_scorer':str(SCORER),'selected_feedback':str(FEEDBACK)},indent=2)+'\n',encoding='utf-8')
BUNDLE=ROOT/'bundle/final-r1'
if not (BUNDLE/'v5_bundle.json').exists(): subprocess.run([sys.executable,str(PACKAGE),'--bundle',str(BUNDLE),'--inherited-base',str(MERGED),'--scorer',str(SCORER),'--feedback',str(FEEDBACK),'--overwrite'],check=True)
assert (BUNDLE/'v5_bundle.json').exists()
SMOKE=EVAL/'smoke'; SMOKE_NAME='apush-frq-grader-v5-smoke'
subprocess.run([sys.executable,str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(DEV),'--output-dir',str(SMOKE),'--model-name',SMOKE_NAME,'--limit','10','--bootstrap-samples','100','--resume'],check=True)
SMOKE_SUMMARY=SMOKE/f'{SMOKE_NAME}_real_summary.json'
smoke=json.loads(SMOKE_SUMMARY.read_text(encoding='utf-8'))
assert smoke['count']==10 and smoke['structured_output_valid_rate']==1.0 and smoke['deterministic_total_rate']==1.0
print({k:smoke[k] for k in ('count','structured_output_valid_rate','deterministic_total_rate','feedback_fallback_rate')})

## 6. Private development comparison, then freeze

Compare prompted base, v4, and v5 on all 60 private development cases. Review only aggregate results. Freezing binds code, datasets, inherited base, adapters, prompts, and hyperparameters; golden evaluation remains blocked until then.

In [ ]:
DEV_OUT=EVAL/'development'; LEGACY_EVAL=REPO/'scripts/eval_hf_model.py'; V5_DEV_NAME='apush-frq-grader-v5-development'
subprocess.run([sys.executable,str(LEGACY_EVAL),'--model',BASE,'--model-name','qwen-base-v5-dev','--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'base'),'--prompt-version','v4','--real-eval','--resume'],check=True)
subprocess.run([sys.executable,str(LEGACY_EVAL),'--model',str(V4),'--model-name','v4-v5-dev','--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'v4'),'--prompt-version','v4','--real-eval','--resume'],check=True)
subprocess.run([sys.executable,str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'v5'),'--model-name',V5_DEV_NAME,'--bootstrap-samples','500','--resume'],check=True)
def first_jsonl(path): return json.loads(path.read_text(encoding='utf-8').splitlines()[0])
comparison={'count':60,'base':first_jsonl(DEV_OUT/'base/qwen-base-v5-dev_real_summary.jsonl'),'v4':first_jsonl(DEV_OUT/'v4/v4-v5-dev_real_summary.jsonl'),'v5':json.loads((DEV_OUT/'v5'/f'{V5_DEV_NAME}_real_summary.json').read_text(encoding='utf-8'))}
DEV_SUMMARY=DEV_OUT/'comparison_summary.json'; DEV_SUMMARY.write_text(json.dumps(comparison,indent=2)+'\n',encoding='utf-8')
assert all(model['count']==60 for model in comparison.values() if isinstance(model,dict))
print(json.dumps(comparison,indent=2))

In [ ]:
FREEZE_CONFIGURATION=True
FROZEN=ROOT/'v5_frozen_config_final-r1.json'
def adapter_weight(path):
 files=list(path.glob('adapter_model.*')); assert len(files)==1,path
 return files[0]
frozen={'run_id':'final-r1','repository_revision':REVISION,'train_sha256':sha(TRAIN),'dev_sha256':sha(DEV),'approval_sha256':sha(APPROVAL),'inherited_base_sha256':merged_meta['inherited_base_sha256'],'scorer_adapter_sha256':sha(adapter_weight(SCORER)),'feedback_adapter_sha256':sha(adapter_weight(FEEDBACK)),'checkpoint_selection_sha256':sha(SELECTION),'bundle_manifest_sha256':sha(BUNDLE/'v5_bundle.json'),'development_summary_sha256':sha(DEV_SUMMARY),'common':COMMON,'tasks':CONFIG,'development_informed':True}
if FREEZE_CONFIGURATION:
 if FROZEN.exists(): assert json.loads(FROZEN.read_text(encoding='utf-8'))==frozen,'Changed configuration needs a new run id'
 else: FROZEN.write_text(json.dumps(frozen,indent=2)+'\n',encoding='utf-8')
assert FROZEN.exists(),'Freeze the reviewed development configuration first.'
print('Frozen:',FROZEN)

## 7. One-time 53-case golden evaluation

Run only after freezing. It must emit aggregate criterion confusion matrices, score distributions, length/reference-total calibration, evidence 0/1/2 errors, and bootstrap confidence intervals. Never retune against these answers.

In [ ]:
GOLD=EVAL/'golden'; GOLD_NAME='apush-frq-grader-v5-final'; FROZEN_CONFIG=FROZEN
assert FROZEN_CONFIG.exists(),'Freeze the reviewed development configuration first.'
GOLD_SUMMARY=GOLD/f'{GOLD_NAME}_real_summary.json'
subprocess.run([sys.executable,str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(REPO/'artifacts/data/eval_cb_cases.jsonl'),'--output-dir',str(GOLD),'--model-name',GOLD_NAME,'--development-informed','--bootstrap-samples','2000','--resume'],check=True)
golden=json.loads(GOLD_SUMMARY.read_text(encoding='utf-8')); gold=golden
assert golden['count']==53 and golden['development_informed'] is True
golden_evaluation_is_development_informed=True
print(json.dumps(golden,indent=2))

## 8. Locked release gates and aggregate-only export

Any failed gate means non-production-ready; do not tune on golden results. The ZIP excludes all JSONL, essays, review packets, prompts/excerpts, and per-case predictions.

In [ ]:
RELEASE_SCRIPT=REPO/'scripts/check_v5_release.py'; GATE=GOLD/'release_gate_receipt.json'
release_run=subprocess.run([sys.executable,str(RELEASE_SCRIPT),'--summary',str(GOLD_SUMMARY),'--output',str(GATE)])
decision=json.loads(GATE.read_text(encoding='utf-8'))
assert (release_run.returncode==0)==decision['release_ready']
decision['policy']='All gates passed.' if decision['release_ready'] else 'Non-production-ready; do not retune on golden answers.'
REPORT=ROOT/'reports/final-r1/v5_run_report.md'; REPORT.parent.mkdir(parents=True,exist_ok=True)
REPORT.write_text(f"""# APUSH FRQ Grader v5 Run Report

Golden evaluation: 53 CB-derived cases (**development-informed**). Capped golden style excerpts shaped synthetic generation; this is not untouched.

| Metric | Result |
|---|---:|
| QWK | {golden['qwk']:.4f} |
| Total MAE | {golden['total_mae']:.4f} |
| Within one | {golden['total_within_one_rate']:.2%} |
| Structured validity | {golden['structured_output_valid_rate']:.2%} |
| Feedback grounding | {golden['evidence_grounding_rate']:.2%} |

Release: **{'PASS' if decision['release_ready'] else 'FAIL — non-production-ready'}**. Do not retune against golden answers.
""",encoding='utf-8')
DIAGNOSTICS=GOLD/f'{GOLD_NAME}_v5_diagnostics.json'; ZIP=ROOT/'v5_report_final-r1.zip'
exports=[REPORT,GOLD_SUMMARY,DIAGNOSTICS,GATE,FROZEN_CONFIG,MERGED_META,SELECTION,BUNDLE/'v5_bundle.json']
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for path in exports:z.write(path,path.name)
with zipfile.ZipFile(ZIP) as z:names=z.namelist()
assert not any(name.endswith('.jsonl') for name in names)
assert not any(x in name.lower() for name in names for x in ('essay','prediction','review_packet'))
print(json.dumps(decision,indent=2)); print('Aggregate-only bundle:',ZIP)
DOWNLOAD_REPORT_BUNDLE=True
if DOWNLOAD_REPORT_BUNDLE:
 from google.colab import files
 files.download(str(ZIP))